In [21]:
import pandas_ta as ta
import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

dataNames =["AAPL","MSFT","AMZN","GOOG","META","TSLA","NVDA","PYPL","INTC","NFLX","ADBE","CSCO","CMCSA","PEP","AVGO","TXN","QCOM","ADP","COST","TMUS"]	
# get datadf = yf.download("AAPL", start = startdate, end = enddate)

In [22]:

data = []
startdate = "2019-01-01"
enddate = "2021-01-01"
# add data of all the indicators to the array from yfinance
for i in range(len(dataNames)):
    df = yf.download(dataNames[i], start = startdate, end = enddate)
    data.append(df)

[*********************100%%**********************]  1 of 1 completed
[*********************100%%**********************]  1 of 1 completed
[*********************100%%**********************]  1 of 1 completed
[*********************100%%**********************]  1 of 1 completed
[*********************100%%**********************]  1 of 1 completed
[*********************100%%**********************]  1 of 1 completed
[*********************100%%**********************]  1 of 1 completed
[*********************100%%**********************]  1 of 1 completed
[*********************100%%**********************]  1 of 1 completed
[*********************100%%**********************]  1 of 1 completed
[*********************100%%**********************]  1 of 1 completed
[*********************100%%**********************]  1 of 1 completed
[*********************100%%**********************]  1 of 1 completed
[*********************100%%**********************]  1 of 1 completed
[*********************100%%*******

In [23]:
#print(df)

In [24]:
def append_indicator_for_startegy(df):
    

    # qqe mod
    qqe_value = ta.qqe(df['Close'])
    # hull suite len = 60 mul = 3
    hull_suite = ta.hma(df['Close'], 60, 3)
    # volume oscillator
    volume_oscillator = ta.pvo(df['Volume'], 12, 26, 9, False)
    # add above to df
    qqe_value = qqe_value[qqe_value.columns[0]]
    volume_oscillator = volume_oscillator[volume_oscillator.columns[0]]


    df['qqe_value'] = qqe_value
    df['hull_suite'] = hull_suite
    df['volume_oscillator'] = volume_oscillator
    
    
    
    return df

In [25]:
def signal_generation(df,volumelimit,followoriginal,qqelimit=50,profitstop=1.05,lossstop=0.95):
    # follow original is a 1 2  1 original 2 duplicate
    # signal generation
    # buy signal
    df['signal'] = 0;#0,-1 sell,1 buy

    #blue in qqe green in hull price above bar
    df.loc[(df['qqe_value'] > qqelimit) & (df['hull_suite'] > df['Close']) & (df['volume_oscillator'] > volumelimit), 'signal'] = 1

    df.loc[(df['qqe_value'] < qqelimit) & (df['hull_suite'] < df['Close']) & (df['volume_oscillator'] < -volumelimit), 'signal'] = -1

    if followoriginal == 2:
        df.loc[(df['qqe_value'] < qqelimit) & (df['hull_suite'] < df['Close']) & (df['volume_oscillator'] < -volumelimit), 'signal'] = -1
        #use this the other one seems a lot of computation or infinite loop
    elif followoriginal == 1:
    # sell signal check after each buy signal see for loss stop or profit stop
        for i in range(len(df)):
            if df['signal'][i] == -1:
                j = i + 1
                while(j<len(df) and df['signal'][j] == 0 and 
                      (df['Close'][j] < df['Close'][i])*profitstop or 
                      df['Close'][i] > df['Close'][i]*lossstop) :
                    j += 1
                if j < len(df):
                    df['signal'][j] = -1
        
    return df
    


In [26]:
def backtest(df, amount):

    profit_percent = 0
    current_amount = amount
    position = 0

    # Iterate over the DataFrame
    for i in range(len(df) - 2):

        # If the buy signal is 1 and you have cash, buy what you can
        if df['signal'][i] == 1 and current_amount > 0:
            buy_price = df['Open'][i + 1]
            quantity = current_amount // buy_price
            current_amount -= quantity * buy_price
            position += quantity

        # If the sell signal is -1 and you have a position, sell your entire position
        elif df['signal'][i + 1] == -1 and position > 0:
            sell_price = df['Close'][i + 2]
            current_amount += sell_price * position
            position = 0

    # Sell any remaining position at the end of the DataFrame
    if position > 0:
        sell_price = df['Close'][i + 2]
        current_amount += sell_price * position

    # Calculate the total profit percent
    total_profit_percent = (current_amount - amount) / amount * 100

    return total_profit_percent

In [27]:
def test(df, amount):
    df1=df.copy()
    df1 = append_indicator_for_startegy(df1)
    df1 = signal_generation(df1,0.1,2,50,1.05,0.95)
    # above parameters are volumelimit,followoriginal(1,2,3),qqelimit=50,profitstop=1.05,lossstop=0.95
    return backtest(df1,amount)
    #df = strategy(df)

In [28]:
def testspecific(df,amount , volumelimit,followoriginal,qqelimit=50,profitstop=1.05,lossstop=0.95):
    df1=df.copy()
    df1 = append_indicator_for_startegy(df1)
    if(followoriginal == 3):
        df1 = signal_generation(df1,volumelimit,1,qqelimit,profitstop,lossstop)
        df1 = signal_generation(df1,volumelimit,2,qqelimit,profitstop,lossstop)

    else:
        df1 = signal_generation(df1,volumelimit,followoriginal,qqelimit,profitstop,lossstop)

    return backtest(df1,amount)

In [29]:
pp = data[0].copy()

pp = append_indicator_for_startegy(pp)
pp = signal_generation(pp,0.1,2,50,1.05,0.95)
print(pp.loc[pp['signal'] == 1])


                  Open        High         Low       Close   Adj Close  \
Date                                                                     
2019-05-07   51.470001   51.855000   50.207500   50.715000   48.949982   
2019-05-08   50.474998   51.334999   50.437500   50.724998   48.959629   
2019-05-09   50.099998   50.419998   49.165001   50.180000   48.433601   
2019-05-10   49.355000   49.712502   48.192501   49.294998   47.762630   
2019-05-13   46.927502   47.369999   45.712502   46.430000   44.986687   
2019-08-01   53.474998   54.507500   51.685001   52.107498   50.487698   
2019-08-02   51.382500   51.607498   50.407501   51.005001   49.419468   
2019-08-05   49.497501   49.662498   48.145000   48.334999   46.832466   
2019-08-06   49.077499   49.517502   48.509998   49.250000   47.719032   
2019-08-07   48.852501   49.889999   48.455002   49.759998   48.213169   
2019-08-08   50.049999   50.882500   49.847500   50.857498   49.276550   
2019-08-09   50.325001   50.689999   4

In [30]:
for i in range(len(data)):
    # test only for valid data
    if len(data[i]) > 0:
        #params
        # df ,amount to be tetsted ,
        # volumelinit lower to place order 
        # followoriginal 1 2 3 1 original 2 duplicate 3 both RECOMMENDED 2
        # qqelimit in qqe value
        # profitstop obv
        # lossstop obv
        prft = testspecific(data[i],100000,0.1,2,50,20,0.95)
        #print name and actual stock loss
        print(dataNames[i],"found",prft,"actual",data[i]['Close'][len(data[i])-1]/data[i]['Close'][0]*100)# obv cant get actual 


AAPL found 48.707149112701416 actual 336.09423500487225
MSFT found 57.3261450958252 actual 219.95647955662596


C:\Users\Acer\AppData\Local\Temp\ipykernel_19760\3128713778.py:11: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  if df['signal'][i] == 1 and current_amount > 0:
C:\Users\Acer\AppData\Local\Temp\ipykernel_19760\3128713778.py:18: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  elif df['signal'][i + 1] == -1 and position > 0:
C:\Users\Acer\AppData\Local\Temp\ipykernel_19760\3128713778.py:12: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  

AMZN found 2.2681743850708007 actual 211.60850938281772
GOOG found -3.3217809104919436 actual 167.5077736516886
META found 0.5063574829101563 actual 201.3266645104029


C:\Users\Acer\AppData\Local\Temp\ipykernel_19760\3128713778.py:11: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  if df['signal'][i] == 1 and current_amount > 0:
C:\Users\Acer\AppData\Local\Temp\ipykernel_19760\3128713778.py:18: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  elif df['signal'][i + 1] == -1 and position > 0:
C:\Users\Acer\AppData\Local\Temp\ipykernel_19760\3128713778.py:12: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  

TSLA found 253.8648611831665 actual 1137.7369394103778
NVDA found 102.23028050994873 actual 383.35046801311125
PYPL found 18.365445114135742 actual 273.1195299687955


...

C:\Users\Acer\AppData\Local\Temp\ipykernel_19760\3128713778.py:11: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  if df['signal'][i] == 1 and current_amount > 0:
C:\Users\Acer\AppData\Local\Temp\ipykernel_19760\3128713778.py:18: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  elif df['signal'][i + 1] == -1 and position > 0:
C:\Users\Acer\AppData\Local\Temp\ipykernel_19760\3128713778.py:12: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  

AVGO found 29.300084716796874 actual 172.71508642303607
TXN found 31.37436321258545 actual 173.7744946384216
QCOM found 101.41365046691895 actual 265.4006834289156


C:\Users\Acer\AppData\Local\Temp\ipykernel_19760\3128713778.py:11: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  if df['signal'][i] == 1 and current_amount > 0:
C:\Users\Acer\AppData\Local\Temp\ipykernel_19760\3128713778.py:18: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  elif df['signal'][i + 1] == -1 and position > 0:
C:\Users\Acer\AppData\Local\Temp\ipykernel_19760\3128713778.py:12: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  

ADP found -7.683792785644532 actual 135.39265235485888
COST found 30.449631591796877 actual 184.01055327568784
TMUS found 11.776634155273436 actual 206.63500105626076


C:\Users\Acer\AppData\Local\Temp\ipykernel_19760\3128713778.py:11: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  if df['signal'][i] == 1 and current_amount > 0:
C:\Users\Acer\AppData\Local\Temp\ipykernel_19760\3128713778.py:18: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  elif df['signal'][i + 1] == -1 and position > 0:
C:\Users\Acer\AppData\Local\Temp\ipykernel_19760\3128713778.py:12: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  